In [2]:
%%html
<style>
    table {
        display: inline-block
    }
</style>

# How to run SIM

## Compiling
------------

### Initial compile
This will compile the code and create the executable `zoupa`.

```bash
cd /where/you/want/your/SIM/
./setup_script
export HDF5_DISABLE_VERSION_CHECK=2
```

### Subsequent compiles
Each time a part of the code is changed in the `*.f90` or `*.h` files in `src/`, the code needs to be recompiled.  
This can be done by simply running:  

```bash
scons-3
```

You may specify another executable name simply by using the `exe` option:

```bash
scons exe='sim01'
```

To clean up object files, type:

```bash
scons --clean   (or scons -c)
```

Note that the compilation can be distributed on `n` processors using the option `-j n`. 

## Inputfile, namelist and forcing
----------------------------------

The `setup_script` linked the default inputfile `input` and the namelist `namelist` into the base directory so that the model can find them and the default forcing can be found in the `forcing` directory. The model should thus run with those default setting without problem. The following paragraphs give some examples of how to adapt inputfiles, namelists and forcing to your needs.

### Inputfile
The inputfile tells the model some basic parameters about the model setup:  

| parameter         | value                 | details                                                  |
| :---------------- | :-------------------: | :------------------------------------------------------- |
| `namelist`        | `0` or `1`            | `1` if a namelist is to be read, `0` otherwise           |
| `restart`         | `0` or `1`            | `1` if a restart is to be read, `0` otherwise            |
| `restart date`    | `YYYY-MM-DD:hh:mm:ss` | only set if `restart==1`, date of the restart to be used |
| `restart version` | `01`, ..., `99`       | only set if `restart==1`, version number of the restart  |
| `exp version`     | `01`, ..., `99`       | two digit version number of the experiment               |
| `starting date`   | `YYYY-MM-DD:hh:mm:ss` | start date of the experiment                             |
| `end date`        | `YYYY-MM-DD:hh:mm:ss` | end date of the experiment                               |
| `posting date`    | `YYYY-MM-DD:hh:mm:ss` | one or more dates at which outputfiles are written       |

A input file may then look like this example:

```bash
1                   input namelist
1                   restart
2008-01-01:00:00:00 restart date      
90                  restart version
91                  exp version
2008-01-01:00:00:00 starting date
2008-01-01:12:00:00 end date
2008-01-01:02:00:00 posting date
2008-01-01:04:00:00 posting date
2008-01-01:06:00:00 posting date
2008-01-01:08:00:00 posting date
2008-01-01:10:00:00 posting date
stop                end of post date string
```

More examples can be found in the `tests/inputfiles/` directory.  
In case you want to include many posting dates (_i.e._ write output variables) at regular intervals, the jupyter notebook [create_input_file](../aux/create_input_file.ipynb) automatically creats inputfiles with just start dates, end dates, and an interval for the outputs provided.  
If using a different inputfile than the default one, you need to provide that new file when running the model. You can, for example, link the new file into the root directory like `ln -sf /inputfiles/new_input input` and then run the code as described below.

### Namelist
The model reads a namelist if `namelist` is set to `1` in the inputfile. A namelist is a way to change model parameters without changing the code directly, and thus without requiring to recompile the code after changing the parameters. The model looks for the namelist in the root directory under the name `namelist`. If you want to use another namelist than the default one in `namelists/`, you need to link your namelist into the root directory `ln -sf namelists/new_namelist namelist`. More examples of namelists can be found in `tests/namelists`.  
The parameters that can be set in namelists are the following:  
<br>

| model parameter       | default value          | options                                                            | details                                                         |
| :-------------------- | :--------------------: | :----------------------------------------------------------------- | :-------------------------------------------------------------- |
| `Dynamic`             | `.true.`               | use sea ice dynamics (`.true.`, `.false.`)                         | momentum equation and rheology are solved if `Dynamic = .true.` |
| `ThermoDyn`           | `.true.`               | use sea ice thermodynamics (`.true.`, `.false.`)                   | h and A are altered due to thermodynamic forcings if `ThermoDyn = .true.` |
| `linearization`       | `'Zhang'`              | linearization method (`'Zhang'`, `'Tremblay'`)                     | ? |
| `regularization`      | `'tanh'`               | regularization method (`'tanh'`, `'Kreyscher'`, `'capping'`)       | regularization method used in [`viscouscoefficient.f90`](../src/viscouscoefficient.f90) |
| `ini_guess`           | `'previous time step'` | initial guess (`'previous time step'`, `'freedrift'`)              | initial guess for `u` and `v` passed to the solver              |
| `adv_scheme`          | `'upwind'`             | advection scheme (`'upwind'`, `'upwindRK2'`, `'semilag'`)          | [advection scheme](documentation.ipynb#Advection-Schemes) for the advection of tracers `h` and `A`       |
| `AirTemp`             | `'MonthlyMean'`        | air temperature (`'MonthlyMean'`, `'specified'`)                   | air temperature used as [forcing](documentation.ipynb#Forcing) for thermodynamics |
| `OcnTemp`             | `'calculated'`         | water temperature (`'calculated'`, `'MonthlyClim'`, `'specified'`) | water temperature used as [forcing](documentation.ipynb#Forcing) for thermodynamics |
| `Wind`                | `'6hours'`             | wind forcing (`'6hours'`, `'clim'`, `'specified'`)                 | wind used as [forcing](documentation.ipynb#Forcing) for dynamics |
| `RampupWind`          | `.false.`              | ramp up wind forcing (`.true.`, `.false.`)                         | if `Wind = 'specified'`, exponentially ramp up wind from 0 to 10 m s^-1 over the course of 6 hours |
| `RampupForcing`       | `.false.`              | ramp up forcing (`.true.`, `.false.`)                              | if `Wind = 'specified'`, ramp up the wind stress by 0.3 n m^-2 per hour |
| `Current`             | `'specified'`          | ocean currents (`'clim'`, `'specified'`)                           | legacy parameter, ocean currents are assumed to be 0 |
| `Periodic_x`          | `0`                    | boundary condition in x-direction (`0`, `1`)                       | if `0`, open boudary condition in `x`, if `1` domain is periodic in `x` |
| `Periodic_y`          | `0`                    | boundary condition in y-direction (`0`, `1`)                       | if `0`, open boudary condition in `y`, if `1` domain is periodic in `y` |
| `ideal`               | `.false.`              | idealized experiment (`.true.`, `.false.`)                         | if `.true.`, Corioilis (`f`) and turning angles (`theta`) are set to `0` |
| `Rheology`            | `1`                    | rheology (`1`, `2`, `3`)                                           | choice of rheology; `1`: ellipse, `2`: triangle, `3`: MEB |
| `IMEX`                | `0`                    | IMplicit-EXplicit integration   (`0`, `1`, `2`)                    | IMplicit-EXplicit integration scheme for dynamics; `0`: split in time (not IMEX), `1`: for Picard solver, `2`: for JFNK solver |
| `BDF`                 | `0`                    | Backwards Difference Formula (`0`, `1`)                            | `adv_scheme = upwindRK2` in combination with `BDF = 1` leads to second-order accuracy in time |
| `visc_method`         | `2`                    | viscous coefficient calculation method (`1`, `2`, `3`, `4`)        | calculate the viscous coefficient based on `1`: [Tremblay and Mysak, 1997](https://journals.ametsoc.org/view/journals/phoc/27/11/1520-0485_1997_027_2342_msiaag_2.0.co_2.xml); `2` [Lemieux et al., 2012](https://www.sciencedirect.com/science/article/pii/S0021999112002665); `3`: [Bouillon et al., 2013](https://www.sciencedirect.com/science/article/pii/S146350031300098X) (modified); `4`: [Bouillon et al., 2013](https://www.sciencedirect.com/science/article/pii/S146350031300098X) |
| `solver`              | `2`                    | solver (`1`, `2`, `3`)                                             | `1`: Picard (*e.g.* [Lemieux and Tremblay, 2009](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2008JC005017)), `2`: JFNK (Jacobian-free Newton–Krylov, *e.g.* [Lemieux et al., 2012](https://www.sciencedirect.com/science/article/pii/S0021999112002665)), `3`: EVP (based on [Hunke, 2001](https://www.sciencedirect.com/science/article/pii/S0021999101967105?via%3Dihub)) |
| `BasalStress`         | `.false.`              | use ice basal stress due to grounding (`.true.`, `.false.`)        |  |

<br><br>


| numerical parameter | default value       | options                                                       | details |
| :--------------     | :-----------------: | :------------------------------------------------------------ | :------ |
| `Deltat`            | `1200d0`            | model time step in seconds                                    | higher resolution requires shorter time steps |
| `gamma_nl`          | `1d-03`             | nonlinear convergence criterion for JFNK solver               | used when `solver = 1`, see [Lemieux et al., 2012](https://www.sciencedirect.com/science/article/pii/S0021999112002665) |
| `NLmax`             | `200`               | maximum number of Newton loop for JFNK solver                 | used when `solver = 1`, see [Lemieux et al., 2012](https://www.sciencedirect.com/science/article/pii/S0021999112002665) |
| `OLmax`             | `500`               | maximum number of outer loop for Picard solver                | used when `solver = 2`, see [Lemieux and Tremblay, 2009](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2008JC005017) |
| `Nsub`              | `120`               | number of EVP sub-cycles                                      | used when `solver = 3`, see [Hunke, 2001](https://www.sciencedirect.com/science/article/pii/S0021999101967105?via%3Dihub) |

<br><br>


| physical parameter  | default value       | options                                                       | details                                  |
| :--------------     | :-----------------: | :------------------------------------------------------------ | :--------------------------------------- |
| `Pstar`             | `27.5d03`           | ice yied stress [N m^-2]                                      | The minimum stress that must be applied to initiate significant flow and a significant drop in viscosity. |
| `C`                 | `20d0`              | ice concentration parameter                                   | Empirical constant characterizing the strong dependence of the compressive strength on sea ice concentration. ([Lemieux et al., 2012](https://www.sciencedirect.com/science/article/pii/S0021999112002665), [Hibler III, 1979](https://journals.ametsoc.org/view/journals/phoc/9/4/1520-0485_1979_009_0815_adtsim_2_0_co_2.xml)) |
| `e_ratio`           | `2.0d0`             | ellipse aspect ratio                                          | aspect ratio of the elliptical yield curve |
| `k1`                | `8d0`               | landfast ice parameter                                        | greater `k1` = easier grounding of ice     |
| `k2`                | `15d0`              | landfast ice parameter                                        | greater `k2` = larger stress due to grounding |
| `rhoair`            | `1.3d0`             | air density [kg m^-3]                                         | used to calculate air-ice stress and heat fluxes |
| `rhoice`            | `9d02`              | ice density [kg m^-3]                                         | important for dynamics and thermodynamics |
| `rhowater`          | `1026d0`            | water density [kg m^-3]                                       | used to calculate ocean-ice stress and heat fluxes |
| `Cdair`             | `1.2d-03`           | air-ice drag coefficient                                      | used to calculate air-ice stress |
| `Cdwater`           | `5.5d-03`           | water-ice drag coefficient                                    | used to calculate ocean-ice stress |
| `f`                 | `1.46d-04`          | Coriolis parameter [s^-1]                                     |  |

<br><br>

### Forcing
To start, the simplest way is to use the example forcing included in the github repository. For more details on how to construct the forcing based on [ERA5](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-complete?tab=overview) reanalysis see [create_simple_forcing.ipynb](../aux/create_simple_forcing.ipynb) and the [documentation](documentation.ipynb).

In case you do not use the example forcing, you need to link your forcing into the forcing directory `/where/you/want/your/SIM/forcing/`. The forcing files need to follow the naming conventions for the model to recognize the files:

| forcing term      | files names                                      | details                                                                        |
| :---------------- | :----------------------------------------------- | :----------------------------------------------------------------------------- |
| wind `6hours`     | `wind_dx<DX>_nx<NX>_ny<NY>.nc`                   | both components of (`u` and `v`) are in the same file and provided on a B-grid |
| wind `clim`       | `climatological_[uv]wnd_dx<DX>_nx<NX>_ny<NY>.nc` | two seperate files for `u` and `v` provided on a B-grid                        |
| air temperature   | `air.mon.mean_dx<DX>_nx<NX>_ny<NY>.nc`           | whole time series of monthly air temperature in one file, provided on a C-grid |
| ocean temperature | `Tocn??_dx<DX>_nx<NX>_ny<NY>`                    | climatological monthly mean ocean temperature in 12 seperate file, one for each month, provided on a C-grid |

with `<DX>` being the horizontal resolution with two decimals, *e.g.* `32.00` or `2.00`, and `<NX>` and `<NY>` being the number of grid cells in `x` and `y` directions, respectively, *e.g.* `160` and `135` for a resolution of 32 km or `2560` and `2160` for a resolution of 2 km.  

## Parameters
-------------

There are some model parameters that cannot be set in a namelist because the compiled code needs to be different for the model to include those changes. Those can be set in `src/parameter.h` and are the following parameters, mostly concerning the grid. (Changing horizontal resolution of the model is probably the most common case in which you need to change any of those)  

| model parameter | default value           | options                                         | details                                                                 |
| :-------------- | :---------------------: | :-----------------------------------------------| :---------------------------------------------------------------------- |
| `nx`            | `160`                   | x-dimension of the domain                       | see `src/parameter.h` for valid combination of `nx`, `ny`, and `Deltax` |
| `ny`            | `135`                   | y-dimension of the domain                       | see `src/parameter.h` for valid combination of `nx`, `ny`, and `Deltax` |
| `Deltax`        | `32d03`                 | grid resolution [m]                             | see `src/parameter.h` for valid combination of `nx`, `ny`, and `Deltax` |
| `ntot`          | `nx * ny`               | total number of grid cells                      | automatically computed                                                  |
| `nvar`          | `(nx+1)\*ny+(ny+1)\*nx` | total number of u,v var cells                   | automatically computed                                                  |
| `uniaxial`      | `.false.`               | uniaxial loading experiment                     | set to`.true.` and choose matching `nx`, `ny`, and `Deltax` for the uniaxial loading experiment |
| `ideal_bridge`  | `.false.`               | ideal ice bridge experiment                     | set to`.true.` and choose matching `nx`, `ny`, and `Deltax` for the ideal ice bridge experiment |
| `beta`          | `32.0d0`                | angle of the grid with respect to Greenwich (0) | counter-clockwise rotation of the grid with respect to `0` degrees loongitude |
| `img`           | `50`                    | rstr value for FGMRES                           | size of Krylov subspace, should not exceed 50 ([see `fgmresD.f90`](../src/fgmresD.f90)) |
| `img1`          | `img + 1`               |                                                 |  |
| `nbuoy`         | `3000`                  | number of buoys                                 | number of buoys to track if `BuoyTrack = .true.` in [`par_get.f90`](../src/par_get.f90) |
| `S0`            | `1340d0`                | Solar constant [W m^-2]                         | solar constant used in [`heatfluxes.f90`](../src/heatfluxes.f90) and [`shortwave.f90`](../src/shortwave.f90) |  

After changing anything in `parameter.h`, the code needs to be recompiled.

## Running the code
-------------------

The following command starts the simulation with `input` as an inputfile and will store the log messages in `outputfile.log`.

```bash
./zoupa < input > outputfile.log &
```

The model output variables are stored in the `output` directory.

## Output analysis
------------------

The output variables are stored as text files in `output/`. One easy way to conduct some analysis and create plots is to convert them to `netcdf` and use python and modules like `xarray`. The jupyter notebook [convert_output2netcdf](../aux/convert_output2netcdf.ipynb) takes care of that. Just set the six parameters `exp`, `dx_str`, `version`, `startdate`, `enddate` and `dt` and the notebook is set up to convert the output files and create a single netcdf file including a functioning time axis as well as x- and y-coordinates for the grid cell centers (`XC`, `YC`) and the sides/corners (`XL`, `YL`). 

## Using github
---------------

You need to have a SSH key set up with github in order to clone the repository. See [the github documentation](https://docs.github.com/en/authentication/connecting-to-github-with-ssh/adding-a-new-ssh-key-to-your-github-account) for details.

The easiest way to create your own SIM repository in which you can commit changes to the code etc. is to [fork](https://docs.github.com/en/pull-requests/how-tos/work-with-forks/fork-a-repo) the [SIM repository](https://github.com/McGill-sea-ice/SIM) and the clone your own repository `git clone git@github.com:<your-own>/SIM.git` (replace `<your-own>` with your namespace on github).  
It is then recommended to create a new branch in which you aplly changes `git checkout -b <new-branch-name>`. Once you're satisfied with the changes you made you might want to feed them back into the main repository, this should be done using a pull request as detailed below.

### Pull requests
Before creating a pull request, you should run the following tests to make sure that your code changes did not lead to the model crashing or altering the results. (Note that some changes indeed aim to change the results of the simulations, in those cases the model should still succeed to run all the tests.)

#### Testing procedure

Create a `TESTS` directory: 
```bash
cd /where/you/want/your/TESTS
mkdir TESTS
cd TESTS
```

Get the master branch and reference solution in `REF` for reference
```bash
git clone git@github.com:McGill-sea-ice/SIM.git REF
cd REF
./testing_script
```

Get your own branch in `NEW` and run the model
```bash
cd ..
git clone git@github.com:<your-own>/SIM.git NEW
cd NEW
git checkout <choose your branch>
./testing_script
```

Compare the results
```bash
./tests/test_results
```

copy result of tests (should get 5 PASS).

If the tests passed or there are valid reasons why the tests did not pass, you can create a pull request for the main repository.

#### Steps for a pull request on github

1. Click on pull request on github.
2. Enter text to describe code modifications and paste result of tests.
3. Assign someone for reviewing the code.
4. Assign `lbtremblay` for approving the pull request.